# Clear vs disruptive subsequences (with t_disruption)

**Data:** `ecei_mc` 160 channels @ 100 kHz — clear from `clear_decimated`, disrupt from `disrupt_decimated`. Windows are decimated 10× to 7813 time steps.

Plot **10 clear** and **10 disruptive** windows in two rows (each subplot = 160×T image). For disruptive windows, a vertical line marks **t_disruption** (when the label switches from 0 to 1).

Run from **soen_fusion_zero**. Paths default to `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/{clear_decimated, disrupt_decimated}`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

nb_dir = Path(os.path.abspath(".")).resolve()
soen_fusion_zero = nb_dir if (nb_dir / "disruptcnn").is_dir() else nb_dir.parent
if (soen_fusion_zero / "disruptcnn").is_dir() and str(soen_fusion_zero) not in sys.path:
    sys.path.insert(0, str(soen_fusion_zero))
os.chdir(soen_fusion_zero)
print("Working directory:", soen_fusion_zero)

In [ ]:
# ecei_mc: 160 ch, 100 kHz; decimate 10× → 7813 time steps per window
DECIMATED_ROOT = os.environ.get("DECIMATED_ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/disrupt_decimated")
CLEAR_ROOT = os.environ.get("CLEAR_ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/clear_decimated")
ROOT = os.environ.get("ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc")
disrupt_file = Path(soen_fusion_zero) / "disruptcnn" / "shots" / "d3d_disrupt_ecei.final.txt"
clear_file = Path(soen_fusion_zero) / "disruptcnn" / "shots" / "d3d_clear_ecei.final.txt"
norm_stats = Path(ROOT) / "norm_stats.npz"
if not norm_stats.exists():
    norm_stats = Path(soen_fusion_zero) / "norm_stats.npz"

from disruptcnn.dataset_original import EceiDatasetOriginal

inner = EceiDatasetOriginal(
    root=ROOT,
    disrupt_file=str(disrupt_file),
    clear_file=str(clear_file) if clear_file.exists() else None,
    flattop_only=True,
    normalize=norm_stats.exists(),
    data_step=10,
    nsub=781300,
    nrecept=30010,
    decimated_root=DECIMATED_ROOT,
    clear_decimated_root=CLEAR_ROOT if Path(CLEAR_ROOT).exists() else None,
    norm_stats_path=str(norm_stats) if norm_stats.exists() else None,
    decimate_extra=10,
)

clear_inds = np.where(~inner.disruptedi)[0]
disrupt_inds = np.where(inner.disruptedi)[0]
print(f"Clear subsequences: {len(clear_inds)}, Disrupt: {len(disrupt_inds)}")
if len(clear_inds) == 0:
    print("No clear shots (disrupt-only dataset). Top row will be empty; only disrupt row shown.")
if len(clear_inds) < 10 or len(disrupt_inds) < 10:
    print("Fewer than 10 of one class; we will show min(10, available).")

In [ ]:
n_show = 10
np.random.seed(42)
n_clear = min(n_show, len(clear_inds))
n_disrupt = min(n_show, len(disrupt_inds))
pick_clear = np.random.choice(len(clear_inds), size=n_clear, replace=False) if n_clear else np.array([], dtype=int)
pick_disrupt = np.random.choice(len(disrupt_inds), size=n_disrupt, replace=False) if n_disrupt else np.array([], dtype=int)
clear_idx = clear_inds[pick_clear] if n_clear else np.array([], dtype=int)
disrupt_idx = disrupt_inds[pick_disrupt] if n_disrupt else np.array([], dtype=int)

# Each element shape (C, T) e.g. (160, 7813)
clear_signals = []
for i in clear_idx:
    X, target, _, _ = inner[i]
    x = X.numpy()  # (C, T)
    if x.ndim == 1:
        x = x[np.newaxis, :]
    clear_signals.append(x)

disrupt_signals = []
t_disrupt_list = []
for i in disrupt_idx:
    X, target, _, _ = inner[i]
    x = X.numpy()
    if x.ndim == 1:
        x = x[np.newaxis, :]
    t = target.numpy().squeeze()
    first_disrupt = int(np.argmax(t > 0.5)) if np.any(t > 0.5) else -1
    disrupt_signals.append(x)
    t_disrupt_list.append(first_disrupt)

# (C, T) e.g. (160, 7813)
C = clear_signals[0].shape[0] if clear_signals else (disrupt_signals[0].shape[0] if disrupt_signals else 160)
T = clear_signals[0].shape[1] if clear_signals else (disrupt_signals[0].shape[1] if disrupt_signals else 7813)
n_cols = max(n_clear, n_disrupt, 1)
print(f"Shape (C, T) = ({C}, {T}), showing {len(clear_signals)} clear, {len(disrupt_signals)} disrupt")

In [ ]:
n_cols = max(len(clear_signals), len(disrupt_signals), 1)
fig, axes = plt.subplots(2, n_cols, figsize=(2.2 * n_cols, 3.5), sharex=True, sharey="row")
if n_cols == 1:
    axes = axes.reshape(2, 1)

for col in range(n_cols):
    ax = axes[0, col]
    if col < len(clear_signals):
        img = clear_signals[col]  # (C, T)
        ax.imshow(img, aspect="auto", extent=[0, T, C, 0], cmap="viridis", interpolation="nearest")
        ax.set_title(f"Clear {col+1}", fontsize=9)
    else:
        ax.set_title("—", fontsize=9)
    ax.set_xticks([])
    if col == 0:
        ax.set_ylabel("ch")
    ax.set_yticks([])

for col in range(n_cols):
    ax = axes[1, col]
    if col < len(disrupt_signals):
        img = disrupt_signals[col]  # (C, T)
        ax.imshow(img, aspect="auto", extent=[0, T, C, 0], cmap="viridis", interpolation="nearest")
        td = t_disrupt_list[col]
        if td >= 0:
            ax.axvline(td, color="red", linestyle="--", linewidth=1, label="t_disrupt" if col == 0 else "")
        ax.set_title(f"Disrupt {col+1}", fontsize=9)
    else:
        ax.set_title("—", fontsize=9)
    ax.set_xlabel("t" if col == n_cols // 2 else "")
    if col == 0:
        ax.set_ylabel("ch")
    ax.set_yticks([])

fig.suptitle("ecei_mc 160×T: Clear (top) vs Disrupt (bottom); red line = t_disruption", y=1.02)
plt.tight_layout()
plt.show()